In [87]:
### Data Quality Assesment
import pandas as pd
import numpy as np
import unicodedata

In [88]:
## Table 1: DataCoSupplyChainDataset
df = pd.read_csv(r"C:\Users\HAMAHANG\OneDrive\Desktop\Data Analysis\End to end projects\DataCo smart supply chain\00_raw_data\DataCoSupplyChainDataset.csv", encoding='latin-1')
display(df.shape)
display(df.dtypes)
display(df.head())

(180519, 53)

Type                              object
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                   object
Late_delivery_risk                 int64
Category Id                        int64
Category Name                     object
Customer City                     object
Customer Country                  object
Customer Email                    object
Customer Fname                    object
Customer Id                        int64
Customer Lname                    object
Customer Password                 object
Customer Segment                  object
Customer State                    object
Customer Street                   object
Customer Zipcode                 float64
Department Id                      int64
Department Name                   object
Latitude                         float64
Longitude                        float64
Market          

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [89]:
#1 converting data types of Date columns into dates
cols_to_convert = ['order date (DateOrders)', 'shipping date (DateOrders)']
df[cols_to_convert] = df[cols_to_convert].apply(pd.to_datetime)
display(df.dtypes)

Type                                     object
Days for shipping (real)                  int64
Days for shipment (scheduled)             int64
Benefit per order                       float64
Sales per customer                      float64
Delivery Status                          object
Late_delivery_risk                        int64
Category Id                               int64
Category Name                            object
Customer City                            object
Customer Country                         object
Customer Email                           object
Customer Fname                           object
Customer Id                               int64
Customer Lname                           object
Customer Password                        object
Customer Segment                         object
Customer State                           object
Customer Street                          object
Customer Zipcode                        float64
Department Id                           

In [90]:
#converting zipcodes from float64 into strings
cols_to_convert = ["Customer Zipcode", "Order Zipcode"]
df[cols_to_convert] = df[cols_to_convert].astype(str)
display(df[cols_to_convert].dtypes)

Customer Zipcode    object
Order Zipcode       object
dtype: object

In [91]:
#2 Checking for odd naming
#multiple city and state names are of differnet alphabetical charechters. we normalize them:
# columns to check
columns_to_clean = ["Order State", "Customer City", "Order City", "Order Country"]

# pattern: Arabic/Persian characters OR non‑Latin characters
pattern = r"[؀-ۿ]|[^A-Za-zÀ-ÖØ-öø-ÿ\s\-]"

# ---- 1. List odd entries ----
for col in columns_to_clean:
    odd_rows = df[df[col].astype(str).str.contains(pattern, regex=True, na=False)]
    
    print(f"\nColumn: {col}")
    print("Odd entries:", len(odd_rows))
    print(odd_rows[[col]])

# ---- 2. Normalization function (remove accents but keep case) ----
def normalize_text(text):
    if not isinstance(text, str):
        return text
    
    # remove accents
    text = ''.join(
        c for c in unicodedata.normalize('NFD', text)
        if unicodedata.category(c) != 'Mn'
    )
    
    # keep capitalization (first letter capitalized)
    return text.strip().title()

# ---- 3. Apply cleaning to all columns ----
for col in columns_to_clean:
    df[col] = df[col].apply(normalize_text)

print("Successfully altered the odd spellings.")
df[columns_to_clean].head()


Column: Order State
Odd entries: 1094
          Order State
108     Lima (ciudad)
1291    Lima (ciudad)
1857    Th? Dô Hà N?i
2264    Th? Dô Hà N?i
2315    Th? Dô Hà N?i
...               ...
180014  Th? Dô Hà N?i
180169   T?nh C?n Th?
180385  Th? Dô Hà N?i
180440   T?nh C?n Th?
180507  Th? Dô Hà N?i

[1094 rows x 1 columns]

Column: Customer City
Odd entries: 0
Empty DataFrame
Columns: [Customer City]
Index: []

Column: Order City
Odd entries: 642
                Order City
620            Kramators'k
871              K'ut'aisi
984     Reggio nell'Emilia
1379     Dniprodzerzhyns'k
1459     Dniprodzerzhyns'k
...                    ...
177727       Wetter (Ruhr)
177728       Wetter (Ruhr)
179936         Coxs B?z?r
180103               Xi'an
180104               Xi'an

[642 rows x 1 columns]

Column: Order Country
Odd entries: 409
             Order Country
267     Myanmar (Birmania)
513     Myanmar (Birmania)
1135    Myanmar (Birmania)
1186    Myanmar (Birmania)
1187    Myanmar (Birman

,Order State,Customer City,Order City,Order Country
0,Java Occidental,Caguas,Bekasi,Indonesia
1,Rajastan,Caguas,Bikaner,India
2,Rajastan,San Jose,Bikaner,India
3,Queensland,Los Angeles,Townsville,Australia
4,Queensland,Caguas,Townsville,Australia


In [92]:
#checking for odd numeric vals
numeric_cols = [
    "Sales",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Total",
    "Product Price"
]

df[numeric_cols].describe()

,Sales,Order Item Quantity,Order Item Discount,Order Item Total,Product Price
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000
mean,203.772096,2.127638,20.664741,183.107609,141.232550
std,132.273077,1.453451,21.800901,120.043670,139.732492
min,9.990000,1.000000,0.000000,7.490000,9.990000
25%,119.980003,1.000000,5.400000,104.379997,50.000000
50%,199.919998,1.000000,14.000000,163.990005,59.990002
75%,299.950012,3.000000,29.990000,247.399994,199.990005
max,1999.989990,5.000000,500.000000,1939.989990,1999.989990


In [93]:
#3 missing values
null_vals = df.isnull().sum()
print(null_vals)

Type                                  0
Days for shipping (real)              0
Days for shipment (scheduled)         0
Benefit per order                     0
Sales per customer                    0
Delivery Status                       0
Late_delivery_risk                    0
Category Id                           0
Category Name                         0
Customer City                         0
Customer Country                      0
Customer Email                        0
Customer Fname                        0
Customer Id                           0
Customer Lname                        8
Customer Password                     0
Customer Segment                      0
Customer State                        0
Customer Street                       0
Customer Zipcode                      0
Department Id                         0
Department Name                       0
Latitude                              0
Longitude                             0
Market                                0


In [94]:
# Order Zipcode and Product Description having missing values is normal 8 last names, 3 customer states and 3 customer zipcodes missing is not an issue either.

In [95]:
#4 duplicate check
df.duplicated().sum()

np.int64(0)

In [96]:
# important key cols duplicate check
df["Order Id"].duplicated().sum()
df["Order Item Id"].duplicated().sum()

np.int64(0)

In [113]:
#checking for uniqe vlaues
df["Order Id"].nunique()

65752

In [114]:
#checking for uniqe vlaues
df["Order Item Id"].nunique()

180519

In [ ]:
#Order Item Id is the grain of this table, since it is unique per each row, so this is the PK of fact_table

In [97]:
# duplicate check - checking if each product name is exactly mapped to one product id
df["Product Card Id"].nunique()
df.groupby("Product Card Id")["Product Name"].nunique()

Product Card Id
19      1
24      1
35      1
37      1
44      1
       ..
1359    1
1360    1
1361    1
1362    1
1363    1
Name: Product Name, Length: 118, dtype: int64

In [98]:
#5 duplicate check for order id and order item id in combination
df.duplicated(subset=["Order Id", "Order Item Id"]).sum()

np.int64(0)

In [99]:
#checkng if there is a difference between number of unique ids and unique product names
(
df.groupby("Product Card Id")
["Product Name"]
.nunique()
.sort_values(ascending=False)
)

Product Card Id
19      1
821     1
897     1
893     1
886     1
       ..
365     1
364     1
359     1
311     1
1363    1
Name: Product Name, Length: 118, dtype: int64

In [100]:
#checking if each product has only one category
(
df.groupby("Product Card Id")
["Category Name"]
.nunique()
.sort_values(ascending=False)
)

Product Card Id
19      1
821     1
897     1
893     1
886     1
       ..
365     1
364     1
359     1
311     1
1363    1
Name: Category Name, Length: 118, dtype: int64

In [101]:
#6 logic check - no negative value in sales, dicount, Order Item Quantity, Sales, Order Item Total and product price
(df[["Sales per customer", "Order Item Discount", "Order Item Quantity", "Sales", "Order Item Total", "Order Item Product Price"]] < 0).sum()

Sales per customer          0
Order Item Discount         0
Order Item Quantity         0
Sales                       0
Order Item Total            0
Order Item Product Price    0
dtype: int64

In [102]:
#Logic check - checling for possible mismatching between delivery status column and late delivery risk column
mismatch_count = (
    (df['Late_delivery_risk'] == 1) & 
    (df['Delivery Status'] != 'Late delivery')
).sum()

print("Mismatch rows:", mismatch_count)


Mismatch rows: 0


In [103]:
#Logic test - id number check(no negative or missing values)
invalid_count1 = (df["Category Id"] <= 0).sum() + df["Category Id"].isnull().sum()
invalid_count2 = (df["Customer Id"] <= 0).sum() + df["Customer Id"].isnull().sum()
invalid_count3 = (df["Department Id"] <= 0).sum() + df["Department Id"].isnull().sum()
invalid_count4 = (df["Order Customer Id"] <= 0).sum() + df["Order Customer Id"].isnull().sum()
invalid_count5 = (df["Order Id"] <= 0).sum() + df["Order Id"].isnull().sum()
invalid_count6 = (df["Order Item Cardprod Id"] <= 0).sum() + df["Order Item Cardprod Id"].isnull().sum()
invalid_count7 = (df["Product Card Id"] <= 0).sum() + df["Product Card Id"].isnull().sum()
invalid_count8 = (df["Product Category Id"] <= 0).sum() + df["Product Category Id"].isnull().sum()
invalid_count = invalid_count1 + invalid_count2 + invalid_count3 + invalid_count4 + invalid_count5 + invalid_count6+ invalid_count7 + invalid_count8
print("invalid ids:", invalid_count)

invalid ids: 0


In [104]:
#Logic test - states two letter check
odd_states = df[df["Customer State"].astype(str).str.len() != 2]
print("Odd state count:", len(odd_states))
print(odd_states[["Customer State"]])

Odd state count: 3
      Customer State
35704          95758
46440          95758
82511          91732


In [105]:
#only 3 are invalid. we still replace them with Nan
df.loc[df["Customer State"].astype(str).str.len() != 2, "Customer State"] = np.nan
print("successfully changed")
odd_states = df[df["Customer State"].astype(str).str.len() != 2]
print("Odd state count:", len(odd_states))
print(odd_states[["Customer State"]])

successfully changed
Odd state count: 3
      Customer State
35704            NaN
46440            NaN
82511            NaN


In [106]:
#Logic test - Shipping date not before Order dtae
date_check = (df["shipping date (DateOrders)"] <= df["order date (DateOrders)"]).sum()
print("invalid shipping dates:", date_check)

invalid shipping dates: 0


In [107]:
#Logic check - no unavaliable items sold
invalid_sold_items = (df["Product Status"] == 1).sum()
print("invalid product status:", invalid_sold_items)

invalid product status: 0


In [108]:
#7 Validity check for some pre-calculated columns
#Order Item Total Validation

#Expected Logic:
#(Product Price × Order Item Quantity) − Order Item Discount = Order Item Total
expected_total = (
    df["Order Item Product Price"] * df["Order Item Quantity"]
    - df["Order Item Discount"]
)

df["difference"] = expected_total - df["Order Item Total"]

invalid_rows = df[abs(df["difference"]) > 0.01]

print("Invalid rows:", len(invalid_rows))
print(invalid_rows[[
    "Order Item Product Price",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Total",
    "difference"
]])


Invalid rows: 1446
        Order Item Product Price  Order Item Quantity  Order Item Discount  \
5                     327.750000                    1            32.779999   
11                    327.750000                    1            59.000000   
16                    327.750000                    1             6.560000   
23                    327.750000                    1            32.779999   
29                    327.750000                    1            59.000000   
...                          ...                  ...                  ...   
179032                 49.980000                    5            37.490002   
179495                 99.989998                    5            50.000000   
179566                 89.989998                    5            45.000000   
179603                 99.989998                    5            50.000000   
179604                 99.989998                    5            50.000000   

        Order Item Total  difference  
5    

In [109]:
#the difference is super minor and we can say that the calculated column is valid

In [110]:
#checking for category validity (only expected terms should exist - checking for misspells and possible unwanted duplicates)
categorical_cols = [
    "Customer Segment",
    "Market",
    "Order Region",
    "Shipping Mode",
    "Order Status",
    "Category Name",
    "Customer City",
    "Customer Country",
    "Customer State"
]

for col in categorical_cols:
    print("\n", col)
    print(df[col].value_counts(dropna=False))


 Customer Segment
Customer Segment
Consumer       93504
Corporate      54789
Home Office    32226
Name: count, dtype: int64

 Market
Market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64

 Order Region
Order Region
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: count, dtype: int64

 Shipping Mode
Shipping Mode
Standard Class    107752
Second Class       35216
First Class   

In [111]:
#saving cleaned data
df.to_csv(r"C:\Users\HAMAHANG\OneDrive\Desktop\Data Analysis\End to end projects\DataCo smart supply chain\01_clean_data\DataCoSupplyChainDataset_clean.csv", index = False)
print("successfuly saved")

successfuly saved
